# Designing the library schema, versioned with Alembic

By the end you can:

- design five tables that hold a library without letting bad data in;
- explain every key and constraint in them;
- create the schema with Alembic, then change it safely with a version history.

**Needs:** Python 3.11+, `psycopg`, `alembic`, `sqlalchemy`, and a running PostgreSQL.

## Setup (Linux)

### 1. Postgres (skip if `psql --version` already works)

```bash
# PGDG repo — gives the current release (18 here)
sudo install -d /usr/share/postgresql-common/pgdg
sudo curl -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc \
  https://www.postgresql.org/media/keys/ACCC4CF8.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] http://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" \
  | sudo tee /etc/apt/sources.list.d/pgdg.list
sudo apt update
sudo apt install postgresql-18 postgresql-client-18 libpq5
```

Ubuntu's own repos (`sudo apt install postgresql`) give an older release and are fine too — adjust the version number below to match.

### 2. Python packages

```bash
uv venv && source .venv/bin/activate
uv pip install "psycopg>=3,<4" alembic sqlalchemy
```

`libpq5` is what psycopg 3 in pure-Python mode drives. It ships with the client package above, so there is no `[binary]` extra to install.

In VS Code: `Ctrl+Shift+P` -> **Python: Select Interpreter** -> `.venv/bin/python`, then **Developer: Reload Window**.

### 3. Role and database

```bash
sudo pg_createcluster 18 main --start
sudo pg_ctlcluster 18 main start
sudo -u postgres createuser -s gamp
sudo -u postgres createdb library -O gamp
sudo -u postgres psql -c "ALTER USER gamp WITH PASSWORD 'gamp';"
```

The connection string lives in cell 6, so there is nothing to export. Change that one line if your role or database is named differently.

Cell 7 drops and recreates all five tables — scratch database only.

If port 5432 is already busy (a Docker container, say), stop it before starting the cluster.

### 4. When you are done

```bash
dropdb library
```

Deleting `.venv` does **not** touch the database. The data lives in `/var/lib/postgresql/18/main`, owned by the server, not by the venv. To remove it too, run `dropdb` above; to wipe the whole cluster, `sudo pg_dropcluster 18 main --stop`.


## Why a schema comes before code

A schema is the set of rules the database enforces whether your Python remembers them or not.

Before writing one, answer five things:

1. What are the **things**? Books, members, loans.
2. What makes each one **unique**?
3. How are they **connected**?
4. What must **never** be true? A loan due before it was borrowed.
5. What **changes**, and what must not.

## Five tables

| Table | Holds | One row is |
|---|---|---|
| `books` | the catalogue | a title, not a physical object |
| `copies` | physical books | one object you can hand over |
| `members` | people who borrow | one person |
| `loans` | who has which copy | one borrow |
| `fines` | money owed | one charge on one loan |

The first split matters most: **a book is not a copy.** Three copies of one title are one row in `books` and three rows in `copies`. Merge them and you cannot say which copy is on loan.

## Four kinds of key

**Primary key** — identifies a row, never empty.

**Foreign key** — points at another table's primary key.

**Unique** — no two rows share it.

**Check** — a rule every row must satisfy.

Keys are not decoration. Each one is a bug the database refuses to store.

In [ ]:
import os
import psycopg

# Local Postgres over its Unix socket: no password needed.
# Change this line if your role or database is named differently.
DEFAULT_URL = "postgresql:///library?host=/var/run/postgresql"
DATABASE_URL = os.environ.get("DATABASE_URL", DEFAULT_URL)

def q(sql: str, params=None, fetch: bool = False):
    """Run one statement. autocommit so DDL takes effect immediately."""
    with psycopg.connect(DATABASE_URL, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            if fetch and cur.description:
                return cur.fetchall()
            return None

        
try:
    version, = q("SELECT version()", fetch=True)[0]
    print("connected:", version.split(",")[0])
except Exception as exc:
    print("Could not connect ->", exc)
    print("Start the server:  sudo pg_ctlcluster 18 main start")
    print("Create the db:     sudo -u postgres createdb library -O $USER")
    print("Then edit DEFAULT_URL in this cell if the role differs.")

connected: PostgreSQL 18.6 (Ubuntu 18.6-1.pgdg24.04+2) on x86_64-pc-linux-gnu


In [4]:
# Every DDL cell below is safe to re-run. Newest table first,
# because a table cannot be dropped while another points at it.
for table in ("fines", "loans", "copies", "members", "books"):
    q(f"DROP TABLE IF EXISTS {table} CASCADE")

print("dropped")

dropped


## `books` — the catalogue

`book_id` is text, not a number: `BK-001` is readable in logs and in support tickets.

`NOT NULL` on title and author because a book without them is not a book. `isbn` is nullable — old donations arrive without one — but unique when present.

In [8]:
q("""
CREATE TABLE books (
    book_id     TEXT PRIMARY KEY,
    title       TEXT NOT NULL,
    author      TEXT NOT NULL,
    isbn        TEXT UNIQUE,
    created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
)
""")

rows = q("SELECT column_name, data_type, is_nullable FROM information_schema.columns "
         "WHERE table_name = 'books' ORDER BY ordinal_position", fetch=True)
for row in rows:
    print(row)

('book_id', 'text', 'NO')
('title', 'text', 'NO')
('author', 'text', 'NO')
('isbn', 'text', 'YES')
('created_at', 'timestamp with time zone', 'NO')


## `members` and `copies`

This is where the book/copy split pays off: each copy points at its title and carries its own status.

`SERIAL` means "give me the next number", so two librarians cannot pick the same `copy_id`.

In [9]:
q("""
CREATE TABLE members (
    member_id   TEXT PRIMARY KEY,
    full_name   TEXT NOT NULL,
    email       TEXT NOT NULL UNIQUE,
    joined_on   DATE NOT NULL DEFAULT CURRENT_DATE
)
""")

q("""
CREATE TABLE copies (
    copy_id   SERIAL PRIMARY KEY,
    book_id   TEXT NOT NULL REFERENCES books(book_id),
    shelf     TEXT,
    status    TEXT NOT NULL DEFAULT 'available'
              CHECK (status IN ('available', 'on_loan', 'lost'))
)
""")
print("members and copies created")

members and copies created


## `loans` and `fines` — the moving parts

Two foreign keys on `loans`: which copy, which member. Plus a check that no loan is due before it starts.

`REFERENCES books(book_id)` on copies means a copy of a book that does not exist is rejected at insert time — the database refuses orphans.

In [10]:
q("""
CREATE TABLE loans (
    loan_id      SERIAL PRIMARY KEY,
    copy_id      INTEGER NOT NULL REFERENCES copies(copy_id),
    member_id    TEXT NOT NULL REFERENCES members(member_id),
    borrowed_on  DATE NOT NULL DEFAULT CURRENT_DATE,
    due_on       DATE NOT NULL,
    returned_on  DATE,
    CHECK (due_on >= borrowed_on)
)
""")

q("""
CREATE TABLE fines (
    fine_id       SERIAL PRIMARY KEY,
    loan_id       INTEGER NOT NULL REFERENCES loans(loan_id),
    amount_cents  INTEGER NOT NULL CHECK (amount_cents >= 0),
    paid_on       DATE
)
""")
print("loans and fines created")

loans and fines created


## The rule that needs an index

Can one copy be on two open loans at once? No. But `UNIQUE(copy_id)` would forbid the same copy ever being borrowed twice, which is wrong.

The rule is narrower: **at most one loan per copy where `returned_on IS NULL`.** PostgreSQL enforces exactly that with a partial unique index.

Read it as: *among the open loans only, each `copy_id` appears once.*

In [11]:
q("""
CREATE UNIQUE INDEX one_open_loan_per_copy
    ON loans (copy_id)
    WHERE returned_on IS NULL
""")
print("partial unique index created")

partial unique index created


## Proof: six bad inserts

Each one below would corrupt the library. The database refuses all six.

In [12]:
q("INSERT INTO books (book_id, title, author) VALUES ('BK-001', 'The Salt Road', 'A. Adeyemi')")
q("INSERT INTO copies (book_id, shelf) VALUES ('BK-001', 'A1')")
q("INSERT INTO members (member_id, full_name, email) VALUES ('MB-01', 'Ada Obi', 'ada@example.test')")
print("seed ok")

bad = {
    "duplicate book id":    "INSERT INTO books (book_id, title, author) VALUES ('BK-001', 'Again', 'X')",
    "copy of unknown book": "INSERT INTO copies (book_id) VALUES ('BK-999')",
    "duplicate email":      "INSERT INTO members (member_id, full_name, email) VALUES ('MB-02', 'Bola', 'ada@example.test')",
    "impossible status":    "INSERT INTO copies (book_id, status) VALUES ('BK-001', 'missing')",
    "due before borrowed":  "INSERT INTO loans (copy_id, member_id, borrowed_on, due_on) VALUES (1, 'MB-01', '2026-02-01', '2026-01-01')",
    "negative fine":        "INSERT INTO fines (loan_id, amount_cents) VALUES (1, -50)",
}

for label, sql in bad.items():
    try:
        q(sql)
        print(f"ACCEPTED (bad!) {label}")
    except psycopg.Error as exc:
        print(f"rejected  {label:<22} {type(exc).__name__}")

seed ok
rejected  duplicate book id      UniqueViolation
rejected  copy of unknown book   ForeignKeyViolation
rejected  duplicate email        UniqueViolation
rejected  impossible status      CheckViolation
rejected  due before borrowed    CheckViolation
rejected  negative fine          CheckViolation


In [13]:
q("INSERT INTO loans (copy_id, member_id, due_on) VALUES (1, 'MB-01', CURRENT_DATE + 14)")
print("first loan created")

try:
    q("INSERT INTO loans (copy_id, member_id, due_on) VALUES (1, 'MB-01', CURRENT_DATE + 14)")
    print("ACCEPTED (bad!) same copy on two open loans")
except psycopg.Error as exc:
    print("rejected  same copy, second open loan  ->", type(exc).__name__)

q("UPDATE loans SET returned_on = CURRENT_DATE WHERE copy_id = 1")
q("INSERT INTO loans (copy_id, member_id, due_on) VALUES (1, 'MB-01', CURRENT_DATE + 14)")
print("after the return, the same copy can be lent again")

first loan created
rejected  same copy, second open loan  -> UniqueViolation
after the return, the same copy can be lent again


The last two lines matter. The index is **partial**, so it only covers *open* loans. Once `returned_on` is set the copy leaves the index, and it can be borrowed again.

## Money is integer cents

`0.1 + 0.2` is not `0.3` in binary floating point, and a fine that is off by a fraction cannot be reconciled.

In [12]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

fine_cents = 250
print(f"fine: {fine_cents // 100}.{fine_cents % 100:02d} naira")
print(f"ten fines: {10 * fine_cents} cents")

0.30000000000000004
False
fine: 2.50 naira
ten fines: 2500 cents


Rule for this project: **store money as `INTEGER` cents**, format for display only at the edge. Never `FLOAT`, never `REAL`.

## Indexes: the other half of the trade

A primary key is indexed automatically. **A foreign key is not.**

So `WHERE member_id = 'MB-01'` reads every loan. Fine on ten rows, not on a million.

In [14]:
q("CREATE INDEX ix_loans_member_id ON loans (member_id)")

for line, in q("EXPLAIN ANALYZE SELECT * FROM loans WHERE member_id = 'MB-01'", fetch=True):
    print(line)

Seq Scan on loans  (cost=0.00..1.02 rows=1 width=52) (actual time=0.016..0.017 rows=2.00 loops=1)
  Filter: (member_id = 'MB-01'::text)
  Buffers: shared hit=1
Planning:
  Buffers: shared hit=105 read=1
Planning Time: 0.638 ms
Execution Time: 0.044 ms


Look for `Seq Scan` (reads every row) versus `Index Scan` (jumps straight there).

An index is not free: every insert and update also writes the index, and each takes disk. Index the columns you filter and join on, not all of them.

## Alembic: git for the schema

You built that schema by hand. That does not survive: no record of what changed, no way to rebuild a teammate's database, no safe path on production.

Alembic stores the schema as numbered **revisions**, each with an `upgrade()` and a `downgrade()`.

```
library/
├── alembic.ini
├── migrations/
│   ├── env.py
│   └── versions/
│       └── a72317edfb3a_library_schema_v1.py
└── app/
    ├── config.py      DATABASE_URL from the environment
    └── models.py      the five tables as Python classes
```

`models.py` is the schema written as Python. Alembic compares those classes against the live database and writes the difference for you.

Three lines in `migrations/env.py` do the work:

1. import your `Base`;
2. `target_metadata = Base.metadata`;
3. set `sqlalchemy.url` from `DATABASE_URL`.

Skip them and autogenerate compares against nothing, then writes an empty migration. That is the most common Alembic mistake.

In [15]:
for command in [
    "alembic init migrations",
    "# edit migrations/env.py, then:",
    "alembic revision --autogenerate -m 'library schema v1'",
    "alembic upgrade head",
    "alembic current",
]:
    print("$", command)

$ alembic init migrations
$ # edit migrations/env.py, then:
$ alembic revision --autogenerate -m 'library schema v1'
$ alembic upgrade head
$ alembic current


The generated file: `upgrade()` creates, `downgrade()` drops in reverse order, so the migration can be undone.

In [17]:
excerpt = """
revision = 'a72317edfb3a'
down_revision = None


def upgrade() -> None:
    op.create_table('books',
        sa.Column('book_id', sa.String(length=16), nullable=False),
        sa.Column('title', sa.Text(), nullable=False),
        sa.PrimaryKeyConstraint('book_id', name=op.f('pk_books')),
    )
    ...
    op.create_index('one_open_loan_per_copy', 'loans', ['copy_id'], unique=True,
                    postgresql_where=sa.text('returned_on IS NULL'))


def downgrade() -> None:
    op.drop_index('one_open_loan_per_copy', table_name='loans')
    op.drop_table('books')
"""
print(excerpt)


revision = 'a72317edfb3a'
down_revision = None


def upgrade() -> None:
    op.create_table('books',
        sa.Column('book_id', sa.String(length=16), nullable=False),
        sa.Column('title', sa.Text(), nullable=False),
        sa.PrimaryKeyConstraint('book_id', name=op.f('pk_books')),
    )
    ...
    op.create_index('one_open_loan_per_copy', 'loans', ['copy_id'], unique=True,
                    postgresql_where=sa.text('returned_on IS NULL'))


def downgrade() -> None:
    op.drop_index('one_open_loan_per_copy', table_name='loans')
    op.drop_table('books')



After `alembic upgrade head`, the database records itself in a one-row table, `alembic_version`.

`alembic current` prints that revision. `alembic history` lists the chain. `downgrade -1` steps back exactly one revision, `downgrade base` removes them all.

## Changing the schema later

Adding a column is the second migration. You change the model, autogenerate, **read the file**, then upgrade.

```
1. add  phone: Mapped[str | None] = mapped_column(String(20))  to Member
2. alembic revision --autogenerate -m "add member phone"
3. read it:   op.add_column('members', sa.Column('phone', sa.String(20)))
4. alembic upgrade head
5. alembic downgrade -1      # prove it can go back
```

Alembic writes a draft. The review is your job.

## Rules for migrations

1. **Never edit a revision that has been applied.** Add a new one.
2. **One change per revision.** A migration you cannot describe in one line is two migrations.
3. **Read every autogenerated file.** It guesses; you decide.
4. **Test the downgrade.** If `downgrade()` fails, you have no way back.
5. **Commit `versions/` to git.** The files are the schema's history.

In [18]:
harbour = """
documents                  chunks
---------                  ------
doc_id        PK           chunk_id    PK
title                      doc_id      FK -> documents
source_path                ordinal     int, page order
content_sha256 UNIQUE      text        the passage
                           embedding   vector(384)   <- pgvector, week 10
"""
print(harbour)
print()
print("content_sha256 is unique, so ingesting the same page twice is refused by the database.")
print("That is the week 2 idempotency rule, enforced at the schema level.")


documents                  chunks
---------                  ------
doc_id        PK           chunk_id    PK
title                      doc_id      FK -> documents
source_path                ordinal     int, page order
content_sha256 UNIQUE      text        the passage
                           embedding   vector(384)   <- pgvector, week 10


content_sha256 is unique, so ingesting the same page twice is refused by the database.
That is the week 2 idempotency rule, enforced at the schema level.


## Summary

| Idea | Rule |
|---|---|
| Book vs copy | one title, many physical copies |
| Primary key | identifies, never null |
| Foreign key | the database refuses orphans |
| Unique | one email, one isbn |
| Check | `amount_cents >= 0`, `due_on >= borrowed_on` |
| Partial unique index | one **open** loan per copy |
| Money | integer cents, never float |
| Index | foreign keys are not indexed for you |
| Alembic | revisions with `upgrade()` and `downgrade()` |

One line: **put the rule in the database where nothing can skip it, and version the database so you can move it forward and back.**